# Agentic RAG: Router-Retriever System with PDF and Web Search

This notebook implements a beginner-friendly multi-agent RAG workflow for the MSAGI course-end project.

The system has two main agents:

1. **Router Agent**  
   Decides whether a user question should use PDF search, web search, or direct LLM reasoning.

2. **Retriever Agent**  
   Executes the selected path and produces a grounded answer.

The project uses:

- CrewAI for agent orchestration
- CrewAI PDFSearchTool for PDF-based retrieval
- TavilySearchResults for web search
- JSON trace logs for reasoning transparency

## 1. Setup

Before running the notebook, the following files must exist:

- `data/course_end_project_problem_statement.pdf`
- `data/attention_is_all_you_need.pdf`
- `.env`

The `.env` file must contain:

```text
OPENAI_API_KEY=your_openai_api_key_here
TAVILY_API_KEY=your_tavily_api_key_here

In [8]:
from pathlib import Path
from datetime import datetime
import json
import os
import textwrap # textwrap is used for formatting text output in the console

from dotenv import load_dotenv

print("Imports loaded successfully.")

Imports loaded successfully.


In [12]:
# The notebook is stored in:
# assignments/02_agentic_rag_router/notebooks/
#
# Therefore:
# - NOTEBOOK_DIR is the current notebook folder
# - PROJECT_DIR is one level above it
# - DATA_DIR contains the PDFs
# - OUTPUTS_DIR will contain the trace log

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    # This fallback helps if VS Code starts the notebook from the repository root.
    PROJECT_DIR = Path("assignments/02_agentic_rag_router").resolve()

DATA_DIR = PROJECT_DIR / "data"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

PROBLEM_STATEMENT_PDF = DATA_DIR / "course_end_project_problem_statement.pdf"
TRANSFORMER_PDF = DATA_DIR / "attention_is_all_you_need.pdf"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook directory:", NOTEBOOK_DIR)
print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Outputs directory:", OUTPUTS_DIR)
print()
print("Problem statement exists:", PROBLEM_STATEMENT_PDF.exists())
print("Transformer paper exists:", TRANSFORMER_PDF.exists())

Notebook directory: c:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\02_agentic_rag_router\notebooks
Project directory: c:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\02_agentic_rag_router
Data directory: c:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\02_agentic_rag_router\data
Outputs directory: c:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\02_agentic_rag_router\outputs

Problem statement exists: True
Transformer paper exists: True


In [19]:
# Load local environment variables from .env
# This keeps secrets outside the notebook and outside Git.
#
# override=True is important because we may edit .env after the notebook kernel has started.

env_path = PROJECT_DIR / ".env"

print("Looking for .env at:")
print(env_path)
print(".env exists:", env_path.exists())

load_dotenv(env_path, override=True)

openai_key = os.getenv("OPENAI_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

# Remove accidental leading/trailing spaces or newlines.
# This does not change the real key content; it only cleans invisible formatting.
if openai_key:
    openai_key = openai_key.strip()
    os.environ["OPENAI_API_KEY"] = openai_key

if tavily_key:
    tavily_key = tavily_key.strip()
    os.environ["TAVILY_API_KEY"] = tavily_key

print("OPENAI_API_KEY loaded:", bool(openai_key))
print("TAVILY_API_KEY loaded:", bool(tavily_key))

if openai_key:
    print("OPENAI_API_KEY length:", len(openai_key))

if tavily_key:
    print("TAVILY_API_KEY length:", len(tavily_key))
    print("TAVILY_API_KEY preview:", tavily_key[:10] + "..." + tavily_key[-4:])
    print("Starts with tvly:", tavily_key.startswith("tvly"))

if not openai_key:
    raise ValueError("OPENAI_API_KEY is missing. Please add it to the local .env file.")

if not tavily_key:
    raise ValueError("TAVILY_API_KEY is missing. Please add it to the local .env file.")

Looking for .env at:
c:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\02_agentic_rag_router\.env
.env exists: True
OPENAI_API_KEY loaded: True
TAVILY_API_KEY loaded: True
OPENAI_API_KEY length: 164
TAVILY_API_KEY length: 58
TAVILY_API_KEY preview: tvly-dev-4...onBG
Starts with tvly: True


In [14]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import PDFSearchTool
from langchain_community.tools.tavily_search import TavilySearchResults

print("CrewAI and retrieval tools imported successfully.")

CrewAI and retrieval tools imported successfully.


In [21]:
from crewai import LLM
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

# We use a small OpenAI model by default to keep the project cost-efficient.
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

llm = LLM(
    model=MODEL_NAME,
    temperature=0
)

print("LLM configured:", MODEL_NAME)

# CrewAI's PDFSearchTool has a safety check for local file paths.
# In this assignment, we intentionally search a local PDF stored inside our project folder.
os.environ["CREWAI_TOOLS_ALLOW_UNSAFE_PATHS"] = "true"

# Use the resolved absolute path so the tool can find the PDF reliably on Windows.
transformer_pdf_path = str(TRANSFORMER_PDF.resolve())

print("Transformer PDF path:")
print(transformer_pdf_path)
print("Transformer PDF exists:", TRANSFORMER_PDF.exists())

# PDFSearchTool searches inside the static Transformer paper.
pdf_search_tool = PDFSearchTool(
    pdf=transformer_pdf_path
)

# TavilySearchResults is named in the assignment instructions.
# We pass the key explicitly through TavilySearchAPIWrapper because the direct Tavily SDK test worked,
# but the default wrapper did not pick up the key reliably.
tavily_api_wrapper = TavilySearchAPIWrapper(
    tavily_api_key=os.environ["TAVILY_API_KEY"]
)

web_search_tool = TavilySearchResults(
    api_wrapper=tavily_api_wrapper,
    max_results=3
)

print("PDF search tool created.")
print("Web search tool created.")

LLM configured: gpt-4o-mini
Transformer PDF path:
C:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\02_agentic_rag_router\data\attention_is_all_you_need.pdf
Transformer PDF exists: True


CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping file path validation for: C:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\02_agentic_rag_router\data\attention_is_all_you_need.pdf


PDF search tool created.
Web search tool created.


In [22]:
# A smoke test is a small test to check whether the most important parts work.
# We test each retrieval tool with one simple query.

pdf_test_query = "What is multi-head attention in the Transformer?"
web_test_query = "What is Tavily search used for in AI agents?"

print("Testing PDF search...")
pdf_test_result = pdf_search_tool.run(pdf_test_query)
print(str(pdf_test_result)[:1000])

print("\n" + "=" * 80 + "\n")

print("Testing web search...")
web_test_result = web_search_tool.invoke({"query": web_test_query})

print("Number of web results:", len(web_test_result))
print(web_test_result[0])

Testing PDF search...
Relevant Content:




Page 4:

Scaled Dot-Product Attention

Multi-Head Attention

Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several

attention layers running in parallel.

of the values, where the weight assigned to each value is computed by a compatibility function of the

query with the corresponding key.

3.2.1

Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of

queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the

query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the

values.

In practice, we compute the attention function on a set of queries simultaneously, packed together

into a matrix Q. The keys and values are also packed together into matrices K and V . We compute

the matrix of outputs as:

Attention(Q, K, V ) = softmax(QKT

√dk

)V

(1)

The 

In [23]:
def print_wrapped(text, width=100):
    """
    Print long text in a readable way inside the notebook.
    """
    text = str(text)
    for paragraph in text.split("\n"):
        if paragraph.strip():
            print(textwrap.fill(paragraph, width=width))
        else:
            print()


def format_web_results(results):
    """
    Convert Tavily web search results into a readable text block.

    TavilySearchResults usually returns a list of dictionaries.
    Each result can contain title, url, content, and score.
    """
    if not results:
        return "No web results returned."

    formatted = []

    for index, result in enumerate(results, start=1):
        title = result.get("title", "No title")
        url = result.get("url", "No URL")
        content = result.get("content", "No content")
        score = result.get("score", "No score")

        formatted.append(
            f"Web result {index}\n"
            f"Title: {title}\n"
            f"URL: {url}\n"
            f"Score: {score}\n"
            f"Content: {content}"
        )

    return "\n\n".join(formatted)


print("Helper functions ready.")

Helper functions ready.


In [24]:
trace_log = []


def add_trace(step, agent, input_text, output_text, metadata=None):
    """
    Store one trace event.

    This helps us show:
    - which agent acted
    - what input it received
    - what output it produced
    - which route/tool was used
    """
    event = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "step": step,
        "agent": agent,
        "input": str(input_text),
        "output": str(output_text),
        "metadata": metadata or {}
    }

    trace_log.append(event)
    return event


def save_trace_log():
    """
    Save all trace events to a local JSON file.
    """
    trace_path = OUTPUTS_DIR / "trace_log.json"

    with open(trace_path, "w", encoding="utf-8") as file:
        json.dump(trace_log, file, indent=2, ensure_ascii=False)

    return trace_path


print("Trace logger ready.")

Trace logger ready.


In [25]:
router_agent = Agent(
    role="Router Agent",
    goal=(
        "Decide the best retrieval path for a user question. "
        "The available paths are pdf_search, web_search, and direct_llm."
    ),
    backstory=(
        "You are a careful routing specialist in an Agentic RAG system. "
        "You do not answer the question yourself. "
        "Your only job is to decide which retrieval path should be used."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

retriever_agent = Agent(
    role="Retriever Agent",
    goal=(
        "Create a clear, source-grounded answer using the retrieval context "
        "provided by the selected tool."
    ),
    backstory=(
        "You are a retrieval specialist. "
        "You receive a user question, the selected route, and retrieved context. "
        "Your job is to answer only from the available context when retrieval was used."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("Router Agent created.")
print("Retriever Agent created.")

Router Agent created.
Retriever Agent created.


In [33]:
def parse_json_from_text(text):
    """
    The Router Agent should return JSON.

    Sometimes LLMs add a short explanation around the JSON.
    This function extracts the JSON object safely.
    """
    text = str(text)
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found in router output.")

    json_text = text[start:end + 1]
    return json.loads(json_text)


def fallback_route(question):
    """
    Simple fallback if the router response cannot be parsed.

    This keeps the notebook robust while still making the AI router
    the main decision-maker.
    """
    question_lower = question.lower()

    pdf_keywords = [
        "transformer",
        "attention",
        "self-attention",
        "multi-head",
        "positional encoding",
        "encoder",
        "decoder",
        "bleu",
        "vaswani"
    ]

    web_keywords = [
        "latest",
        "current",
        "today",
        "recent",
        "news",
        "2025",
        "2026",
        "web",
        "online",
        "tavily"
    ]

    if any(keyword in question_lower for keyword in web_keywords):
        return {
            "route": "web_search",
            "reason": "The question appears to ask for current or external information."
        }

    if any(keyword in question_lower for keyword in pdf_keywords):
        return {
            "route": "pdf_search",
            "reason": "The question appears related to the Transformer paper."
        }

    return {
        "route": "direct_llm",
        "reason": "The question appears to be general and does not clearly require PDF or web retrieval."
    }


async def route_question(question):
    """
    Ask the Router Agent to choose the best route.

    This is async because Jupyter notebooks already run inside an event loop.
    CrewAI therefore needs kickoff_async() instead of kickoff().
    """
    routing_task = Task(
        description=f"""
You are the Router Agent.

Classify the user question into exactly one route:

1. pdf_search
Use this when the question asks about the provided Transformer paper,
including attention, self-attention, multi-head attention, positional encoding,
encoder-decoder architecture, BLEU results, training setup, or paper-specific claims.

2. web_search
Use this when the question asks for current, recent, external, online, tool-specific,
or changing information.

3. direct_llm
Use this only when the question is general and does not need PDF or web retrieval.

Return ONLY valid JSON in this exact structure:
{{
  "route": "pdf_search | web_search | direct_llm",
  "reason": "brief reason"
}}

User question:
{question}
""",
        expected_output="Valid JSON with route and reason.",
        agent=router_agent
    )

    routing_crew = Crew(
        agents=[router_agent],
        tasks=[routing_task],
        process=Process.sequential,
        verbose=False
    )

    raw_result = await routing_crew.kickoff_async()
    raw_text = str(raw_result)

    try:
        decision = parse_json_from_text(raw_text)
    except Exception as error:
        decision = fallback_route(question)
        decision["parser_warning"] = str(error)
        decision["raw_router_output"] = raw_text

    allowed_routes = {"pdf_search", "web_search", "direct_llm"}

    if decision.get("route") not in allowed_routes:
        fallback_decision = fallback_route(question)
        fallback_decision["router_warning"] = f"Invalid route returned: {decision.get('route')}"
        fallback_decision["raw_router_output"] = raw_text
        decision = fallback_decision

    add_trace(
        step="route_question",
        agent="Router Agent",
        input_text=question,
        output_text=decision,
        metadata={"raw_router_output": raw_text}
    )

    return decision


print("Async router function ready.")

Async router function ready.


In [28]:
def retrieve_context(question, route):
    """
    Execute the selected retrieval path.

    The Router Agent chooses the route.
    This function runs the matching retrieval tool.
    """
    if route == "pdf_search":
        raw_result = pdf_search_tool.run(question)
        context = str(raw_result)

        metadata = {
            "tool": "PDFSearchTool",
            "source": str(TRANSFORMER_PDF.name)
        }

    elif route == "web_search":
        raw_result = web_search_tool.invoke({"query": question})
        context = format_web_results(raw_result)

        metadata = {
            "tool": "TavilySearchResults",
            "source": "web_search",
            "result_count": len(raw_result) if isinstance(raw_result, list) else None
        }

    elif route == "direct_llm":
        context = (
            "No external retrieval was used. "
            "The Retriever Agent may answer from general model knowledge."
        )

        metadata = {
            "tool": "none",
            "source": "direct_llm"
        }

    else:
        raise ValueError(f"Unknown route: {route}")

    add_trace(
        step="retrieve_context",
        agent="Retriever Agent",
        input_text={"question": question, "route": route},
        output_text=context,
        metadata=metadata
    )

    return context, metadata


print("Retrieval function ready.")

Retrieval function ready.


In [34]:
async def generate_answer(question, route, context, retrieval_metadata):
    """
    Ask the Retriever Agent to produce the final answer from the retrieved context.

    This is async because Jupyter notebooks already run inside an event loop.
    CrewAI therefore needs kickoff_async() instead of kickoff().
    """
    answer_task = Task(
        description=f"""
You are the Retriever Agent.

User question:
{question}

Selected route:
{route}

Retrieval metadata:
{json.dumps(retrieval_metadata, indent=2)}

Retrieved context:
{context}

Instructions:
- Answer the user question clearly and concisely.
- If route is pdf_search, base your answer on the PDF context.
- If route is web_search, base your answer on the web search results.
- If route is direct_llm, answer from general knowledge.
- If the retrieved context is insufficient, say what is missing.
- End with a short "Source path" line that says which route/tool was used.
""",
        expected_output="A clear answer with a short source path line.",
        agent=retriever_agent
    )

    answer_crew = Crew(
        agents=[retriever_agent],
        tasks=[answer_task],
        process=Process.sequential,
        verbose=False
    )

    result = await answer_crew.kickoff_async()
    answer = str(result)

    add_trace(
        step="generate_answer",
        agent="Retriever Agent",
        input_text={
            "question": question,
            "route": route,
            "retrieval_metadata": retrieval_metadata
        },
        output_text=answer
    )

    return answer


print("Async answer generation function ready.")

Async answer generation function ready.


In [35]:
async def answer_question(question):
    """
    Full Agentic RAG workflow:

    1. Receive user question
    2. Router Agent selects route
    3. Retriever path gets context
    4. Retriever Agent writes final answer
    5. Trace log is saved
    """
    print("=" * 100)
    print("USER QUESTION")
    print("=" * 100)
    print_wrapped(question)

    decision = await route_question(question)
    route = decision["route"]

    print("=" * 100)
    print("ROUTER DECISION")
    print("=" * 100)
    print(json.dumps(decision, indent=2))

    context, retrieval_metadata = retrieve_context(question, route)

    print("=" * 100)
    print("RETRIEVAL METADATA")
    print("=" * 100)
    print(json.dumps(retrieval_metadata, indent=2))

    answer = await generate_answer(question, route, context, retrieval_metadata)

    print("=" * 100)
    print("FINAL ANSWER")
    print("=" * 100)
    print_wrapped(answer)

    trace_path = save_trace_log()

    print("=" * 100)
    print("TRACE LOG SAVED")
    print("=" * 100)
    print(trace_path)

    return {
        "question": question,
        "router_decision": decision,
        "retrieval_metadata": retrieval_metadata,
        "answer": answer,
        "trace_path": str(trace_path)
    }


print("Async end-to-end workflow ready.")

Async end-to-end workflow ready.


In [37]:
pdf_demo = await answer_question(
    "According to the Transformer paper, why is self-attention useful compared with recurrent layers?"
)

USER QUESTION
According to the Transformer paper, why is self-attention useful compared with recurrent layers?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are the Router Agent.                                                                                      │
│                                                                                                                 │
│  Classify the user question into exactly one route:                                                             │
│                                                                                                                 │
│  1. pdf_search                                                                                                  │
│  Use this when the question asks about the provided Transformer paper,                                          │
│  including attention, self-attention, multi-head attention, positional encoding,                                │
│  encoder-decoder architecture, BLEU results, training setup, or paper-specific claims.                          │
│                                                                                                                 │
│  2. web_search                                                                                                  │
│  Use this when the question asks for current, recent, external, online, tool-specific,                          │
│  or changing information.                                                                                       │
│                                                                                                                 │
│  3. direct_llm                                                                                                  │
│  Use this only when the question is general and does not need PDF or web retrieval.                             │
│                                                                                                                 │
│  Return ONLY valid JSON in this exact structure:                                                                │
│  {                                                                                                              │
│    "route": "pdf_search | web_search | direct_llm",                                                             │
│    "reason": "brief reason"                                                                                     │
│  }                                                                                                              │
│                                                                                                                 │
│  User question:                                                                                                 │
│  According to the Transformer paper, why is self-attention useful compared with recurrent layers?               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "route": "pdf_search",                                                                                       │
│    "reason": "The question specifically asks about a concept discussed in the Transformer paper."               │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ROUTER DECISION
{
  "route": "pdf_search",
  "reason": "The question specifically asks about a concept discussed in the Transformer paper."
}
RETRIEVAL METADATA
{
  "tool": "PDFSearchTool",
  "source": "attention_is_all_you_need.pdf"
}


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever Agent                                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are the Retriever Agent.                                                                                   │
│                                                                                                                 │
│  User question:                                                                                                 │
│  According to the Transformer paper, why is self-attention useful compared with recurrent layers?               │
│                                                                                                                 │
│  Selected route:                                                                                                │
│  pdf_search                                                                                                     │
│                                                                                                                 │
│  Retrieval metadata:                                                                                            │
│  {                                                                                                              │
│    "tool": "PDFSearchTool",                                                                                     │
│    "source": "attention_is_all_you_need.pdf"                                                                    │
│  }                                                                                                              │
│                                                                                                                 │
│  Retrieved context:                                                                                             │
│  Relevant Content:                                                                                              │
│                                                                                                                 │
│  corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We              │
│                                                                                                                 │
│  chose this function because we hypothesized it would allow the model to easily learn to attend by              │
│                                                                                                                 │
│  relative positions, since for any fixed offset k, PEpos+k can be represented as a linear function of           │
│                                                                                                                 │
│  PEpos.                                                                                                         │
│                                                                                                                 │
│  We also experimented with using learned positional embeddings [9] instead, and found that the two              │
│                                                                                                                 │
│  versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version              │
│                                                                                                                 │
│  because it may allow the model to extrapolate to seque

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever Agent                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Self-attention is useful compared to recurrent layers for several reasons outlined in the Transformer paper.   │
│  Firstly, self-attention allows for a total computational complexity per layer that is more efficient,          │
│  particularly when the sequence length \( n \) is smaller than the representation dimensionality \( d \). In    │
│  contrast, recurrent layers require \( O(n) \) sequential operations, which limits parallelization.             │
│                                                                                                                 │
│  Self-attention connects all positions with a constant number of sequentially executed operations, while        │
│  recurrent layers require \( O(n) \) sequential operations. This means that self-attention can significantly    │
│  reduce the path length between long-range dependencies, making it easier to learn such dependencies, which is  │
│  a key challenge in many sequence transduction tasks.                                                           │
│                                                                                                                 │
│  Moreover, self-attention can be parallelized more effectively, allowing for faster computation, especially in  │
│  tasks involving long sequences. The paper also notes that self-attention could yield more interpretable        │
│  models, as it allows for inspection of attention distributions.                                                │
│                                                                                                                 │
│  In summary, self-attention provides advantages in computational efficiency, parallelization, and the ability   │
│  to learn long-range dependencies more effectively than recurrent layers.                                       │
│                                                                                                                 │
│  Source path: pdf_search                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

FINAL ANSWER
Self-attention is useful compared to recurrent layers for several reasons outlined in the
Transformer paper. Firstly, self-attention allows for a total computational complexity per layer
that is more efficient, particularly when the sequence length \( n \) is smaller than the
representation dimensionality \( d \). In contrast, recurrent layers require \( O(n) \) sequential
operations, which limits parallelization.

Self-attention connects all positions with a constant number of sequentially executed operations,
while recurrent layers require \( O(n) \) sequential operations. This means that self-attention can
significantly reduce the path length between long-range dependencies, making it easier to learn such
dependencies, which is a key challenge in many sequence transduction tasks.

Moreover, self-attention can be parallelized more effectively, allowing for faster computation,
especially in tasks involving long sequences. The paper also notes that self-attention could yie

In [38]:
web_demo = await answer_question(
    "According to current Tavily documentation, what is Tavily Search designed to do for AI agents and RAG applications?"
)

USER QUESTION
According to current Tavily documentation, what is Tavily Search designed to do for AI agents and
RAG applications?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are the Router Agent.                                                                                      │
│                                                                                                                 │
│  Classify the user question into exactly one route:                                                             │
│                                                                                                                 │
│  1. pdf_search                                                                                                  │
│  Use this when the question asks about the provided Transformer paper,                                          │
│  including attention, self-attention, multi-head attention, positional encoding,                                │
│  encoder-decoder architecture, BLEU results, training setup, or paper-specific claims.                          │
│                                                                                                                 │
│  2. web_search                                                                                                  │
│  Use this when the question asks for current, recent, external, online, tool-specific,                          │
│  or changing information.                                                                                       │
│                                                                                                                 │
│  3. direct_llm                                                                                                  │
│  Use this only when the question is general and does not need PDF or web retrieval.                             │
│                                                                                                                 │
│  Return ONLY valid JSON in this exact structure:                                                                │
│  {                                                                                                              │
│    "route": "pdf_search | web_search | direct_llm",                                                             │
│    "reason": "brief reason"                                                                                     │
│  }                                                                                                              │
│                                                                                                                 │
│  User question:                                                                                                 │
│  According to current Tavily documentation, what is Tavily Search designed to do for AI agents and RAG          │
│  applications?                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "route": "web_search",                                                                                       │
│    "reason": "The question asks for current information about Tavily Search, which requires online retrieval."  │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ROUTER DECISION
{
  "route": "web_search",
  "reason": "The question asks for current information about Tavily Search, which requires online retrieval."
}
RETRIEVAL METADATA
{
  "tool": "TavilySearchResults",
  "source": "web_search",
  "result_count": 3
}


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever Agent                                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are the Retriever Agent.                                                                                   │
│                                                                                                                 │
│  User question:                                                                                                 │
│  According to current Tavily documentation, what is Tavily Search designed to do for AI agents and RAG          │
│  applications?                                                                                                  │
│                                                                                                                 │
│  Selected route:                                                                                                │
│  web_search                                                                                                     │
│                                                                                                                 │
│  Retrieval metadata:                                                                                            │
│  {                                                                                                              │
│    "tool": "TavilySearchResults",                                                                               │
│    "source": "web_search",                                                                                      │
│    "result_count": 3                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
│  Retrieved context:                                                                                             │
│  Web result 1                                                                                                   │
│  Title: Tavily AI · GitHub                                                                                      │
│  URL: https://github.com/tavily-ai                                                                              │
│  Score: 0.7942736                                                                                               │
│  Content: ### 🕸️ Crawl entire domains at scale                                                                  │
│                                                                                                                 │
│  Optimized for intelligent URL decisions and results. Navigate complex websites with smart crawling strategies  │
│  that respect robots.txt, handle dynamic content, and efficiently discover all accessible pages.                │
│                                                                                                                 │
│  Built specifically for AI agents and RAG workflows.                                                            │
│                                                                                                                 │
│  ## Popular repositories Loading                                                                                │
│                                                        

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever Agent                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Tavily Search is designed specifically for AI agents and RAG (Retrieval-Augmented Generation) applications.    │
│  It provides fast, reliable APIs that enable developers to search the web for real-time, contextual             │
│  information. The tool optimizes for relevancy and low latency, allowing for intelligent content extraction     │
│  and ranking that prioritizes the most relevant results for AI applications. Additionally, Tavily Search        │
│  supports structured data extraction from URLs, transforming webpages into LLM-ready data with automatic        │
│  content cleaning and format conversion.                                                                        │
│                                                                                                                 │
│  Overall, Tavily Search enhances the capabilities of AI agents by enabling them to efficiently gather and       │
│  utilize web-sourced insights and citations.                                                                    │
│                                                                                                                 │
│  Source path: web_search                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

FINAL ANSWER
Tavily Search is designed specifically for AI agents and RAG (Retrieval-Augmented Generation)
applications. It provides fast, reliable APIs that enable developers to search the web for real-
time, contextual information. The tool optimizes for relevancy and low latency, allowing for
intelligent content extraction and ranking that prioritizes the most relevant results for AI
applications. Additionally, Tavily Search supports structured data extraction from URLs,
transforming webpages into LLM-ready data with automatic content cleaning and format conversion.

Overall, Tavily Search enhances the capabilities of AI agents by enabling them to efficiently gather
and utilize web-sourced insights and citations.

Source path: web_search
TRACE LOG SAVED
c:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\02_agentic_rag_router\outputs\trace_log.json


In [39]:
direct_demo = await answer_question(
    "In simple terms, what is the difference between a router agent and a retriever agent?"
)

USER QUESTION
In simple terms, what is the difference between a router agent and a retriever agent?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are the Router Agent.                                                                                      │
│                                                                                                                 │
│  Classify the user question into exactly one route:                                                             │
│                                                                                                                 │
│  1. pdf_search                                                                                                  │
│  Use this when the question asks about the provided Transformer paper,                                          │
│  including attention, self-attention, multi-head attention, positional encoding,                                │
│  encoder-decoder architecture, BLEU results, training setup, or paper-specific claims.                          │
│                                                                                                                 │
│  2. web_search                                                                                                  │
│  Use this when the question asks for current, recent, external, online, tool-specific,                          │
│  or changing information.                                                                                       │
│                                                                                                                 │
│  3. direct_llm                                                                                                  │
│  Use this only when the question is general and does not need PDF or web retrieval.                             │
│                                                                                                                 │
│  Return ONLY valid JSON in this exact structure:                                                                │
│  {                                                                                                              │
│    "route": "pdf_search | web_search | direct_llm",                                                             │
│    "reason": "brief reason"                                                                                     │
│  }                                                                                                              │
│                                                                                                                 │
│  User question:                                                                                                 │
│  In simple terms, what is the difference between a router agent and a retriever agent?                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "route": "direct_llm",                                                                                       │
│    "reason": "The question is general and does not require PDF or web retrieval."                               │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ROUTER DECISION
{
  "route": "direct_llm",
  "reason": "The question is general and does not require PDF or web retrieval."
}
RETRIEVAL METADATA
{
  "tool": "none",
  "source": "direct_llm"
}


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever Agent                                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are the Retriever Agent.                                                                                   │
│                                                                                                                 │
│  User question:                                                                                                 │
│  In simple terms, what is the difference between a router agent and a retriever agent?                          │
│                                                                                                                 │
│  Selected route:                                                                                                │
│  direct_llm                                                                                                     │
│                                                                                                                 │
│  Retrieval metadata:                                                                                            │
│  {                                                                                                              │
│    "tool": "none",                                                                                              │
│    "source": "direct_llm"                                                                                       │
│  }                                                                                                              │
│                                                                                                                 │
│  Retrieved context:                                                                                             │
│  No external retrieval was used. The Retriever Agent may answer from general model knowledge.                   │
│                                                                                                                 │
│  Instructions:                                                                                                  │
│  - Answer the user question clearly and concisely.                                                              │
│  - If route is pdf_search, base your answer on the PDF context.                                                 │
│  - If route is web_search, base your answer on the web search results.                                          │
│  - If route is direct_llm, answer from general knowledge.                                                       │
│  - If the retrieved context is insufficient, say what is missing.                                               │
│  - End with a short "Source path" line that says which route/tool was used.                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever Agent                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  In simple terms, a router agent is responsible for directing data packets between different networks,          │
│  ensuring that they reach their intended destination efficiently. It operates at the network layer of the OSI   │
│  model and makes decisions based on IP addresses.                                                               │
│                                                                                                                 │
│  On the other hand, a retriever agent is designed to fetch and retrieve specific information or data from a     │
│  database or a set of documents. It focuses on locating and delivering relevant content based on user queries.  │
│                                                                                                                 │
│  In summary, the key difference lies in their functions: router agents manage data traffic between networks,    │
│  while retriever agents focus on information retrieval.                                                         │
│                                                                                                                 │
│  Source path: direct_llm                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

FINAL ANSWER
In simple terms, a router agent is responsible for directing data packets between different
networks, ensuring that they reach their intended destination efficiently. It operates at the
network layer of the OSI model and makes decisions based on IP addresses.

On the other hand, a retriever agent is designed to fetch and retrieve specific information or data
from a database or a set of documents. It focuses on locating and delivering relevant content based
on user queries.

In summary, the key difference lies in their functions: router agents manage data traffic between
networks, while retriever agents focus on information retrieval.

Source path: direct_llm
TRACE LOG SAVED
c:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\02_agentic_rag_router\outputs\trace_log.json


In [40]:
print(f"Number of trace events: {len(trace_log)}")

for event in trace_log:
    print("=" * 100)
    print("Step:", event["step"])
    print("Agent:", event["agent"])
    print("Timestamp:", event["timestamp"])
    print("Metadata:", event["metadata"])
    print("Output preview:")
    print_wrapped(str(event["output"])[:1000])

Number of trace events: 9
Step: route_question
Agent: Router Agent
Timestamp: 2026-07-04T15:34:00
Metadata: {'raw_router_output': '{\n  "route": "pdf_search",\n  "reason": "The question specifically asks about a concept discussed in the Transformer paper."\n}'}
Output preview:
{'route': 'pdf_search', 'reason': 'The question specifically asks about a concept discussed in the
Transformer paper.'}
Step: retrieve_context
Agent: Retriever Agent
Timestamp: 2026-07-04T15:34:02
Metadata: {'tool': 'PDFSearchTool', 'source': 'attention_is_all_you_need.pdf'}
Output preview:
Relevant Content:

corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We

chose this function because we hypothesized it would allow the model to easily learn to attend by

relative positions, since for any fixed offset k, PEpos+k can be represented as a linear function of

PEpos.

We also experimented with using learned positional embeddings [9] instead, and found that the two

vers

In [41]:
trace_path = save_trace_log()

print("Trace log saved here:")
print(trace_path)

print("\nFile exists:", trace_path.exists())
print("File size in bytes:", trace_path.stat().st_size)

Trace log saved here:
c:\Users\ChristopherCopony\Projekte\MSAGI-COURSE\assignments\02_agentic_rag_router\outputs\trace_log.json

File exists: True
File size in bytes: 35153


## 7. Architecture Summary

This notebook implements a simple Agentic RAG system with two main agents:

### Router Agent

The Router Agent receives the user question and decides which path should be used:

- `pdf_search` for questions about the provided Transformer paper
- `web_search` for current, external, or tool-specific online information
- `direct_llm` for general conceptual questions that do not need retrieval

### Retriever Agent

The Retriever Agent receives:

- the original user question
- the selected route
- the retrieved context
- retrieval metadata

It then creates the final answer and includes a short source path.

### Retrieval Tools

The project uses two retrieval tools:

- `PDFSearchTool` from CrewAI for semantic search inside the provided PDF
- `TavilySearchResults` for web search

### Trace Logging

Each major step is written into a trace log:

1. Router decision
2. Retrieval execution
3. Final answer generation

The trace log is saved as `outputs/trace_log.json` and is also displayed inside the notebook.

## 8. Challenges and Trade-offs

### 1. Local PDF path safety check

CrewAI's `PDFSearchTool` blocked the local PDF path at first.

Because this assignment intentionally uses a local PDF stored inside the project folder, the notebook explicitly sets the following environment variable before creating the PDF search tool:

    os.environ["CREWAI_TOOLS_ALLOW_UNSAFE_PATHS"] = "true"

For this local learning project, this is acceptable because the PDF is part of the controlled assignment folder. In a production system, file access should be handled more strictly.

### 2. Tavily wrapper authentication

The direct Tavily Python SDK test worked, but the default LangChain `TavilySearchResults` wrapper initially returned a `401 Unauthorized` error.

This showed that the API key itself was valid, but the wrapper did not reliably pick it up from the environment in the local notebook setup.

The solution was to pass the Tavily API key explicitly through `TavilySearchAPIWrapper`.

This keeps the implementation aligned with the assignment requirement to use `TavilySearchResults`, while making the notebook more reliable.

### 3. CrewAI inside Jupyter

CrewAI's normal `crew.kickoff()` call is synchronous.

VS Code and Jupyter already run inside an event loop. Because of that, the synchronous CrewAI call caused a runtime error.

The notebook therefore uses the async version instead:

    await crew.kickoff_async()

The demo cells also call the workflow with `await`.

### 4. Simplicity over production complexity

The system is intentionally simple and beginner-friendly.

It does not include advanced production features such as:

- persistent vector stores
- user authentication
- monitoring dashboards
- complex fallback chains
- deployment infrastructure

This keeps the implementation focused on the course requirements: agent roles, routing, retrieval, orchestration, trace logging, and a working notebook demo.

### 5. Routing fallback

The Router Agent is the main decision-maker.

However, the notebook includes a small fallback routing function in case the Router Agent returns invalid JSON or an unexpected route.

This improves robustness without turning the project into an overly complex production system.

## 9. Conclusion

This notebook demonstrates a working Agentic RAG Router-Retriever system.

The implementation satisfies the core assignment requirements:

- accepts natural language questions
- uses a Router Agent to choose the retrieval path
- uses a Retriever Agent to generate grounded answers
- retrieves from both a local PDF and the web
- logs agent interactions for traceability
- provides a working notebook demo and README

The project shows how agentic orchestration can combine static domain knowledge with dynamic web search in one coordinated workflow.